<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [6]:

using System;
using System.Collections.Generic;

// ============================================================
// 1. КЛАСС ПОЗИЦИИ ФАКТУРЫ
// ============================================================

public class LineItem
{
    public string Name { get; set; }
    public double Price { get; set; }
    public int Quantity { get; set; }
    public string Category { get; set; }
    public string Description { get; set; }
    public string Manufacturer { get; set; }

    public LineItem(string name, double price)
    {
        Name = name;
        Price = price;
        Quantity = 1;
        Category = "Общее";
        Description = "";
        Manufacturer = "Не указан";
    }

    public LineItem(string name, double price, int quantity, string category)
        : this(name, price)
    {
        Quantity = quantity;
        Category = category;
    }

    public double GetTotalPrice()
    {
        return Price * Quantity;
    }

    public void ChangePrice(double newPrice)
    {
        Price = newPrice;
    }

    public void ChangeQuantity(int quantity)
    {
        Quantity = quantity;
    }

    public void DisplayItem()
    {
        Console.WriteLine(
            $"[{Category}] {Name} — {Quantity} шт. x {Price} руб. = {GetTotalPrice()} руб.");
    }

    // Перегрузка метода DisplayItem
    public void DisplayItem(bool showDescription)
    {
        DisplayItem();

        if (showDescription)
        {
            Console.WriteLine($"Описание: {Description}");
        }
    }
}

// ============================================================
// 2. БАЗОВЫЙ КЛАСС ФАКТУРЫ
// ============================================================

public class Invoice
{
    public string InvoiceNumber { get; set; }
    public DateTime IssueDate { get; set; }
    public double TotalAmount { get; set; }
    public List<LineItem> Lines { get; set; }

    public string CustomerName { get; set; }
    public bool IsPaid { get; set; }
    public double TaxRate { get; set; }
    public string Currency { get; set; }

    public string Status { get; set; }
    public string PaymentMethod { get; set; }

    public Invoice(
        string invoiceNumber,
        DateTime issueDate,
        string customerName = "Не указан")
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
        CustomerName = customerName;
        IsPaid = false;
        TaxRate = 0.20;
        Currency = "руб.";
        TotalAmount = 0;
        Lines = new List<LineItem>();
        Status = "Создана";
        PaymentMethod = "Не указан";
    }

    // Виртуальный метод для полиморфизма
    public virtual void CalculateTotal()
    {
        TotalAmount = 0;

        foreach (LineItem line in Lines)
        {
            TotalAmount += line.GetTotalPrice();
        }

        Console.WriteLine(
            $"Сумма фактуры №{InvoiceNumber}: {TotalAmount:F2} {Currency}");
    }

    // Перегрузка CalculateTotal
    public void CalculateTotal(double discountPercent)
    {
        CalculateTotal();

        if (discountPercent >= 0 && discountPercent <= 100)
        {
            double discount = TotalAmount * discountPercent / 100;
            TotalAmount -= discount;

            Console.WriteLine(
                $"Скидка {discountPercent}%: -{discount:F2} {Currency}");
            Console.WriteLine(
                $"Сумма со скидкой: {TotalAmount:F2} {Currency}");
        }
        else
        {
            Console.WriteLine("Некорректный размер скидки.");
        }
    }

    // Еще одна перегрузка CalculateTotal
    public void CalculateTotal(double discountPercent, bool includeTax)
    {
        CalculateTotal(discountPercent);

        if (includeTax)
        {
            double tax = TotalAmount * TaxRate;
            Console.WriteLine($"НДС ({TaxRate * 100}%): {tax:F2} {Currency}");
        }
    }

    // Виртуальный метод для перекрытия в производных классах
    public virtual void PrintInfo()
    {
        Console.WriteLine($"Фактура №{InvoiceNumber}");
        Console.WriteLine($"Дата: {IssueDate:dd.MM.yyyy}");
        Console.WriteLine($"Заказчик: {CustomerName}");
        Console.WriteLine($"Статус: {Status}");
        Console.WriteLine($"Оплата: {(IsPaid ? "Оплачено" : "Не оплачено")}");
        Console.WriteLine($"Сумма: {TotalAmount:F2} {Currency}");
        Console.WriteLine("Позиции:");

        foreach (LineItem line in Lines)
        {
            line.DisplayItem();
        }
    }

    public virtual void AddLine(LineItem lineItem)
    {
        Lines.Add(lineItem);
        Console.WriteLine($"Добавлена позиция: {lineItem.Name}");
    }

    // Перегрузка AddLine
    public void AddLine(string name, double price)
    {
        AddLine(new LineItem(name, price));
    }

    // Еще одна перегрузка AddLine
    public void AddLine(
        string name,
        double price,
        int quantity,
        string category)
    {
        AddLine(new LineItem(name, price, quantity, category));
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (Lines.Remove(lineItem))
        {
            Console.WriteLine($"Удалена позиция: {lineItem.Name}");
        }
        else
        {
            Console.WriteLine("Такая позиция не найдена.");
        }
    }

    public void MarkAsPaid(string paymentMethod)
    {
        IsPaid = true;
        PaymentMethod = paymentMethod;
        Status = "Оплачена";

        Console.WriteLine(
            $"Фактура №{InvoiceNumber} оплачена. Способ: {PaymentMethod}");
    }

    public int GetLineCount()
    {
        return Lines.Count;
    }

    public LineItem FindLine(string name)
    {
        return Lines.Find(line => line.Name == name);
    }

    public void ChangeCustomer(string customerName)
    {
        CustomerName = customerName;
    }
}

// ============================================================
// 3. ТОВАРНАЯ ФАКТУРА
// ============================================================

public class GoodsInvoice : Invoice
{
    public DateTime SupplyDate { get; set; }
    public string DeliveryAddress { get; set; }
    public double WeightKg { get; set; }
    public string CarrierCompany { get; set; }

    public string Supplier { get; set; }
    public bool Delivered { get; set; }

    public GoodsInvoice(
        string invoiceNumber,
        DateTime issueDate,
        DateTime supplyDate,
        string customerName = "Не указан",
        string deliveryAddress = "Самовывоз",
        double weightKg = 0,
        string carrierCompany = "Служба доставки")
        : base(invoiceNumber, issueDate, customerName)
    {
        SupplyDate = supplyDate;
        DeliveryAddress = deliveryAddress;
        WeightKg = weightKg;
        CarrierCompany = carrierCompany;
        Supplier = "Не указан";
        Delivered = false;
    }

    public override void AddLine(LineItem lineItem)
    {
        base.AddLine(lineItem);
        Console.WriteLine($"Дата поставки: {SupplyDate:dd.MM.yyyy}");
    }

    public override void PrintInfo()
    {
        Console.WriteLine("\n=== ТОВАРНАЯ ФАКТУРА ===");
        base.PrintInfo();
        Console.WriteLine($"Поставщик: {Supplier}");
        Console.WriteLine($"Дата поставки: {SupplyDate:dd.MM.yyyy}");
        Console.WriteLine($"Адрес доставки: {DeliveryAddress}");
        Console.WriteLine($"Вес: {WeightKg} кг");
        Console.WriteLine($"Перевозчик: {CarrierCompany}");
        Console.WriteLine($"Доставлено: {(Delivered ? "Да" : "Нет")}");
    }

    public double CalculateDeliveryCost()
    {
        return CalculateDeliveryCost(50);
    }

    // Перегрузка CalculateDeliveryCost
    public double CalculateDeliveryCost(double ratePerKg)
    {
        double cost = WeightKg * ratePerKg;
        Console.WriteLine($"Стоимость доставки: {cost:F2} {Currency}");
        return cost;
    }

    public void Deliver()
    {
        Delivered = true;
        Console.WriteLine($"Товарная фактура №{InvoiceNumber}: доставка выполнена.");
    }

    public void ChangeAddress(string address)
    {
        DeliveryAddress = address;
    }

    public void ChangeSupplier(string supplier)
    {
        Supplier = supplier;
    }
}

// ============================================================
// 4. ФАКТУРА ЗА УСЛУГИ
// ============================================================

public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }
    public string Performer { get; set; }
    public double DurationHours { get; set; }
    public string ActNumber { get; set; }

    public string ServiceType { get; set; }
    public int QualityRating { get; set; }

    public ServiceInvoice(
        string invoiceNumber,
        DateTime issueDate,
        DateTime serviceDate,
        string customerName = "Не указан",
        string performer = "Сервисный центр",
        double durationHours = 1,
        string actNumber = "ACT-000")
        : base(invoiceNumber, issueDate, customerName)
    {
        ServiceDate = serviceDate;
        Performer = performer;
        DurationHours = durationHours;
        ActNumber = actNumber;
        ServiceType = "Общая услуга";
        QualityRating = 0;
    }

    public override void RemoveLine(LineItem lineItem)
    {
        base.RemoveLine(lineItem);
        Console.WriteLine("Позиция услуги обработана для удаления.");
    }

    public override void PrintInfo()
    {
        Console.WriteLine("\n=== ФАКТУРА ЗА УСЛУГИ ===");
        base.PrintInfo();
        Console.WriteLine($"Дата услуги: {ServiceDate:dd.MM.yyyy}");
        Console.WriteLine($"Исполнитель: {Performer}");
        Console.WriteLine($"Продолжительность: {DurationHours} ч.");
        Console.WriteLine($"Номер акта: {ActNumber}");
        Console.WriteLine($"Тип услуги: {ServiceType}");
        Console.WriteLine($"Оценка качества: {QualityRating}/5");
    }

    public void CompleteService()
    {
        Console.WriteLine($"Услуга по фактуре №{InvoiceNumber} выполнена.");
    }

    // Перегрузка CompleteService
    public void CompleteService(string clientFeedback)
    {
        CompleteService();
        Console.WriteLine($"Отзыв клиента: {clientFeedback}");
    }

    // Еще одна перегрузка CompleteService
    public void CompleteService(int rating, string clientFeedback)
    {
        CompleteService(clientFeedback);
        QualityRating = rating;
        Console.WriteLine($"Оценка качества: {QualityRating}/5");
    }

    public void ChangePerformer(string performer)
    {
        Performer = performer;
    }

    public void ChangeServiceDate(DateTime date)
    {
        ServiceDate = date;
    }

    public void SetServiceType(string serviceType)
    {
        ServiceType = serviceType;
    }
}

// ============================================================
// 5. КОМБИНИРОВАННАЯ ФАКТУРА
// ============================================================

public class CombinedInvoice : Invoice
{
    public bool ReturnAllowed { get; set; }
    public int WarrantyPeriodMonths { get; set; }
    public string ManagerContact { get; set; }
    public string SpecialTerms { get; set; }

    public double ReturnedAmount { get; set; }
    public string ContractNumber { get; set; }

    public CombinedInvoice(
        string invoiceNumber,
        DateTime issueDate,
        bool returnAllowed,
        string customerName = "Не указан",
        int warrantyPeriodMonths = 12,
        string managerContact = "manager@company.ru")
        : base(invoiceNumber, issueDate, customerName)
    {
        ReturnAllowed = returnAllowed;
        WarrantyPeriodMonths = warrantyPeriodMonths;
        ManagerContact = managerContact;
        SpecialTerms = "Стандартная гарантия и обслуживание";
        ReturnedAmount = 0;
        ContractNumber = "Не указан";
    }

    public override void CalculateTotal()
    {
        base.CalculateTotal();

        if (ReturnAllowed)
        {
            Console.WriteLine("Возврат разрешен.");
        }
        else
        {
            Console.WriteLine("Возврат не разрешен.");
        }
    }

    public override void PrintInfo()
    {
        Console.WriteLine("\n=== КОМБИНИРОВАННАЯ ФАКТУРА ===");
        base.PrintInfo();
        Console.WriteLine($"Возврат разрешен: {(ReturnAllowed ? "Да" : "Нет")}");
        Console.WriteLine($"Гарантия: {WarrantyPeriodMonths} мес.");
        Console.WriteLine($"Менеджер: {ManagerContact}");
        Console.WriteLine($"Особые условия: {SpecialTerms}");
        Console.WriteLine($"Номер договора: {ContractNumber}");
        Console.WriteLine($"Сумма возвратов: {ReturnedAmount:F2} {Currency}");
    }

    public void ShowItemFromInvoice(LineItem item)
    {
        if (Lines.Contains(item))
        {
            Console.WriteLine(
                $"В фактуре №{InvoiceNumber} есть позиция '{item.Name}', " +
                $"стоимость: {item.GetTotalPrice()} {Currency}");
        }
        else
        {
            Console.WriteLine("Позиция не найдена в фактуре.");
        }
    }

    public void ExtendWarranty(int additionalMonths)
    {
        if (additionalMonths > 0)
        {
            WarrantyPeriodMonths += additionalMonths;
            Console.WriteLine(
                $"Гарантия продлена. Новый срок: {WarrantyPeriodMonths} мес.");
        }
    }

    // Перегрузка ExtendWarranty
    public void ExtendWarranty(int additionalMonths, double extraFee)
    {
        ExtendWarranty(additionalMonths);

        if (extraFee >= 0)
        {
            TotalAmount += extraFee;
            Console.WriteLine($"Доплата за гарантию: {extraFee} {Currency}");
        }
    }

    public void ProcessReturn(double amount)
    {
        double availableAmount = TotalAmount - ReturnedAmount;

        if (ReturnAllowed && amount > 0 && amount <= availableAmount)
        {
            ReturnedAmount += amount;
            Console.WriteLine($"Возврат оформлен: {amount} {Currency}");
        }
        else
        {
            Console.WriteLine("Возврат невозможен: проверьте сумму и условия.");
        }
    }

    public void ChangeSpecialTerms(string terms)
    {
        SpecialTerms = terms;
    }

    public void ChangeManager(string contact)
    {
        ManagerContact = contact;
    }
}

// ============================================================
// 6. ОБОБЩЁННЫЙ (GENERIC) КЛАСС
// ============================================================

public class InvoiceRegistry<T> where T : Invoice
{
    private List<T> _invoices = new List<T>();

    public string RegistryName { get; set; }

    public InvoiceRegistry(string registryName)
    {
        RegistryName = registryName;
    }

    public void Add(T invoice)
    {
        _invoices.Add(invoice);
        Console.WriteLine(
            $"[Реестр {RegistryName}] Добавлена фактура №{invoice.InvoiceNumber}");
    }

    public void Remove(T invoice)
    {
        if (_invoices.Remove(invoice))
        {
            Console.WriteLine(
                $"[Реестр {RegistryName}] Удалена фактура №{invoice.InvoiceNumber}");
        }
    }

    public void DisplayInvoices()
    {
        Console.WriteLine($"\n===== {RegistryName} =====");

        foreach (T invoice in _invoices)
        {
            // Полиморфный вызов PrintInfo()
            invoice.PrintInfo();
        }
    }

    public double GetGrandTotal()
    {
        double sum = 0;

        foreach (T invoice in _invoices)
        {
            sum += invoice.TotalAmount;
        }

        return sum;
    }

    public int GetCount()
    {
        return _invoices.Count;
    }
}

// ============================================================
// 7. ДЕМОНСТРАЦИЯ РАБОТЫ ПРОГРАММЫ
// ============================================================

Console.OutputEncoding = System.Text.Encoding.UTF8;

Console.WriteLine("===== 1. ПЕРЕГРУЗКА МЕТОДОВ =====");

LineItem product = new LineItem("Ноутбук", 50000);
LineItem service = new LineItem(
    "Настройка программы", 5000, 2, "Услуги");

GoodsInvoice goodsInvoice = new GoodsInvoice(
    "G-001",
    new DateTime(2026, 9, 12),
    new DateTime(2026, 9, 15),
    "ООО Альфа",
    "г. Москва, ул. Ленина, д. 10",
    3.5,
    "СДЭК");

goodsInvoice.AddLine(product);
goodsInvoice.AddLine("Мышь", 1500);
goodsInvoice.AddLine("Кабель HDMI", 600, 2, "Аксессуары");

goodsInvoice.CalculateTotal();
goodsInvoice.CalculateTotal(5);
goodsInvoice.CalculateTotal(5, true);

goodsInvoice.CalculateDeliveryCost();
goodsInvoice.CalculateDeliveryCost(75);

Console.WriteLine("\n===== 2. ФАКТУРА ЗА УСЛУГИ =====");

ServiceInvoice serviceInvoice = new ServiceInvoice(
    "S-001",
    DateTime.Today,
    DateTime.Today,
    "ИП Иванов",
    "Алексей Смирнов",
    2.5,
    "ACT-2026/45");

serviceInvoice.AddLine(service);
serviceInvoice.CalculateTotal();
serviceInvoice.CompleteService();
serviceInvoice.CompleteService("Работа выполнена качественно.");
serviceInvoice.CompleteService(5, "Работа выполнена быстро и качественно.");

Console.WriteLine("\n===== 3. КОМБИНИРОВАННАЯ ФАКТУРА =====");

CombinedInvoice combinedInvoice = new CombinedInvoice(
    "C-001",
    DateTime.Today,
    true,
    "ПАО ТехноСистемы",
    12);

combinedInvoice.AddLine(product);
combinedInvoice.AddLine(service);
combinedInvoice.CalculateTotal();

combinedInvoice.ExtendWarranty(12);
combinedInvoice.ExtendWarranty(6, 3500);

combinedInvoice.ShowItemFromInvoice(product);
combinedInvoice.ProcessReturn(5000);

Console.WriteLine("\n===== 4. ПОЛИМОРФИЗМ С ПЕРЕКРЫТИЕМ =====");

// Список базового типа Invoice содержит разные производные классы.
List<Invoice> invoiceList = new List<Invoice>
{
    goodsInvoice,
    serviceInvoice,
    combinedInvoice
};

// Для каждого объекта вызывается его собственная версия PrintInfo().
foreach (Invoice invoice in invoiceList)
{
    invoice.PrintInfo();
}

Console.WriteLine("\n===== 5. GENERIC-КЛАСС =====");

// Реестр только для товарных фактур.
InvoiceRegistry<GoodsInvoice> goodsRegistry =
    new InvoiceRegistry<GoodsInvoice>("Реестр товарных фактур");

goodsRegistry.Add(goodsInvoice);
goodsRegistry.DisplayInvoices();

Console.WriteLine($"Количество товарных фактур: {goodsRegistry.GetCount()}");

// Реестр для любых фактур и их производных классов.
InvoiceRegistry<Invoice> generalRegistry =
    new InvoiceRegistry<Invoice>("Главный архив бухгалтерии");

generalRegistry.Add(goodsInvoice);
generalRegistry.Add(serviceInvoice);
generalRegistry.Add(combinedInvoice);

generalRegistry.DisplayInvoices();

Console.WriteLine(
    $"Количество фактур в архиве: {generalRegistry.GetCount()}");

Console.WriteLine(
    $"Общая сумма фактур в архиве: {generalRegistry.GetGrandTotal():F2} руб.");

Console.WriteLine("\n===== 6. ДОПОЛНИТЕЛЬНЫЕ МЕТОДЫ =====");

goodsInvoice.Deliver();
goodsInvoice.MarkAsPaid("Банковская карта");
goodsInvoice.ChangeAddress("г. Тюмень, ул. Республики, д. 1");

Console.WriteLine($"Количество позиций: {goodsInvoice.GetLineCount()}");

LineItem foundItem = goodsInvoice.FindLine("Ноутбук");

if (foundItem != null)
{
    foundItem.DisplayItem();
}


===== 1. ПЕРЕГРУЗКА МЕТОДОВ =====
Добавлена позиция: Ноутбук
Дата поставки: 15.09.2026
Добавлена позиция: Мышь
Дата поставки: 15.09.2026
Добавлена позиция: Кабель HDMI
Дата поставки: 15.09.2026
Сумма фактуры №G-001: 52700,00 руб.
Сумма фактуры №G-001: 52700,00 руб.
Скидка 5%: -2635,00 руб.
Сумма со скидкой: 50065,00 руб.
Сумма фактуры №G-001: 52700,00 руб.
Скидка 5%: -2635,00 руб.
Сумма со скидкой: 50065,00 руб.
НДС (20%): 10013,00 руб.
Стоимость доставки: 175,00 руб.
Стоимость доставки: 262,50 руб.

===== 2. ФАКТУРА ЗА УСЛУГИ =====
Добавлена позиция: Настройка программы
Сумма фактуры №S-001: 10000,00 руб.
Услуга по фактуре №S-001 выполнена.
Услуга по фактуре №S-001 выполнена.
Отзыв клиента: Работа выполнена качественно.
Услуга по фактуре №S-001 выполнена.
Отзыв клиента: Работа выполнена быстро и качественно.
Оценка качества: 5/5

===== 3. КОМБИНИРОВАННАЯ ФАКТУРА =====
Добавлена позиция: Ноутбук
Добавлена позиция: Настройка программы
Сумма фактуры №C-001: 60000,00 руб.
Возврат разрешен